# Bokeh 的图形模型

这一章说明 Bokeh 的图形(glyph)模型。

- 模型的属性与 `Spec`
- 散点图、圆、线段
- 多条折线、多边形
- 六角形格子、文字、图像

In [1]:
import helper.magics
from bokeh.plotting import figure
import pandas as pd
import numpy as np
from bokeh.models import ColumnDataSource
from bokeh.layouts import column, row
from bokeh.io import output_notebook, show, reset_output

output_notebook()

Loading BokehJS ...

## 模型的属性

图形的类也从 `Model` 派生，但通常不自己创建图形对象，而是调用 `figure` 的方法：`figure.line()` 做出 `Line`、`figure.scatter()` 做出 `Scatter`，并加到图里。

图形的属性可以连到数据源的列上，例如散点图的点的颜色可以由某一列决定。用本书的 `show_glyph_info()` 能看到属性的类型，下面的程序输出 `Scatter` 的属性。

In [2]:
from bokeh.models import Scatter
from helper.bokeh import show_glyph_info

%omit 40 show_glyph_info(Scatter)

angle: 
------
AngleSpec(units_default="rad")

angle_units: 
------------
NotSerialized(Enum("deg", "rad", "grad", "turn"))

decorations: 
------------
List(Instance(Decoration))

defs: 
-----
Dict(Regex("^@.*$"), Instance(CustomJS))

fill_alpha: 
-----------
AlphaSpec(
    String,
    Float,
    Instance(Value),
    Instance(Field),
    Instance(Expr),
    Struct(value=Float, transform=Instance(Transform)),
    Struct(field=String, transform=Instance(Transform)),
    Struct(expr=Instance(Expression), transform=Instance(Transform)),
)

fill_color: 
-----------
ColorSpec(
    String,
    Nullable(Color),
    Instance(Value),
    Instance(Field),
    Instance(Expr),
    Struct(value=Nullable(Color), transform=Instance(Transform)),
    Struct(field=String, transform=Instance(Transform)),
    Struct(expr=Instance(Expression), transform=Instance(Transform)),
    ...

填充色 `fill_color` 的类型像下面这样。

```python
ColorSpec(
    String,
    Nullable(Color),
    Instance(Value),
    Instance(Field),
    Instance(Expr),
    Struct(value=Nullable(Color), transform=Instance(Transform)),
    Struct(field=String, transform=Instance(Transform)),
    Struct(expr=Instance(Expression), transform=Instance(Transform)),
)
```

类型名里带 `Spec` 的就能连到数据源，值可以是几种类型之一：字符串 `String`、颜色 `Color`、数据源的列 `Field` 等。

颜色的类型 `Color` 接受表示颜色的字符串或元组，也可以直接写常见的颜色名。

### 散点图(`Scatter`)

`Scatter` 是散点图的模型，下面这些属性都能连到数据源。

In [3]:
from bokeh.models import Scatter

Scatter.dataspecs().keys()

dict_keys(['x', 'y', 'size', 'angle', 'line_color', 'line_alpha', 'line_width', 'line_join', 'line_cap', 'line_dash', 'line_dash_offset', 'fill_color', 'fill_alpha', 'hatch_color', 'hatch_alpha', 'hatch_scale', 'hatch_pattern', 'hatch_weight', 'marker'])

Bokeh 的枚举值放在 `bokeh.core.enums` 里，例如 `marker` 的候选可以用下面的代码查到。

In [4]:
from bokeh.core import enums

%col 4 list(enums.MarkerType)

['asterisk',             'circle',               'circle_cross',         'circle_dot',          
 'circle_x',             'circle_y',             'cross',                'dash',                
 'diamond',              'diamond_cross',        'diamond_dot',          'dot',                 
 'hex',                  'hex_dot',              'inverted_triangle',    'plus',                
 'square',               'square_cross',         'square_dot',           'square_pin',          
 'square_x',             'star',                 'star_dot',             'triangle',            
 'triangle_dot',         'triangle_pin',         'x',                    'y']                   


散点图里不一定把颜色与形状本身放进数据源，那时就用变换器把数据变成颜色或形状。下面的例子按 3 个品种分别画鸢尾花花瓣的大小。

❶ `factor_mark()` 把分类的值变成点的形状，参数是列名、形状的列表与分类的列表。❷ `factor_cmap()` 把分类变成颜色，这里从 `Category10_10` 里取 3 种颜色。❸ 给 `scatter()` 传 `legend_field`，就用那一列做出图例；传 `color` 会同时设好 `line_color` 与 `fill_color`，传 `alpha` 则会同时设好 `line_alpha` 与 `fill_alpha`。

In [5]:
from bokeh.transform import factor_mark, factor_cmap
from bokeh.palettes import Category10_10
from random import sample

df = pd.read_csv("data/iris.csv")
fig = figure(width=500, height=350)
source = ColumnDataSource(df)
species_types = df.species.unique().tolist()
marker = factor_mark("species", ["diamond", "circle", "triangle"], species_types)  # ❶
color = factor_cmap(
    "species", sample(Category10_10, k=len(species_types)), species_types
)  # ❷
scatter = fig.scatter(
    x="sepal_length",
    y="sepal_width",
    legend_field="species",
    marker=marker,
    color=color,
    source=source,
    size=12,
    alpha=0.6,
)  # ❸
show(fig)

### 圆(`Circle`)

`Circle` 可以用数据坐标的单位来决定半径，因此放大时圆也跟着变大。与半径有关的属性有 3 个。

| 属性  | 类型  | 说明  |
|:--|:--|:--|
| `radius`  | `DistanceSpec`  | 圆的半径 |
| `radius_dimension`  | `Enum`  | 半径按哪个轴的单位来量：`'x'`、`'y'`、`'max'`、`'min'` |
| `radius_units`  | `Enum`  | 半径的单位，`'screen'` 或 `'data'`，默认是 `'data'` |

`radius_units='data'` 再加上 `match_aspect=True` 时，x 轴与 y 轴的单位长度相同，半径就是数据的长度；轴的缩尺不同时用 `radius_dimension` 选：`'x'` 用 x 轴、`'y'` 用 y 轴、`'max'` 用大的那个、`'min'` 用小的那个的单位。

下面的例子用大小与颜色各不相同的圆组成一幅画：`venus-face.csv` 的每一行是一个圆，`x`、`y` 是中心、`radius` 是半径、`r`、`g`、`b` 是颜色的分量；用 `read_csv()` 读进来之后，用 `assign()` 做出把 `r`、`g`、`b` 变成十六进制颜色的 `color` 列。

In [6]:
df = pd.read_csv("data/venus-face.csv")
df2 = df.assign(
    color=df.eval("r * 2**16 + g * 2**8 + b").apply("#{:06x}".format),
).drop(["r", "g", "b"], axis=1)
%C df.head();df2.head()

             df.head()                         df2.head()         
-----------------------------------  -----------------------------
     x    y   radius    r    g    b       x    y   radius    color
0    0  500  71.5124  197  212  213  0    0  500  71.5124  #c5d4d5
1    0  363  63.9851  191  209  211  1    0  363  63.9851  #bfd1d3
2    0  139  59.2101  201  215  213  2    0  139  59.2101  #c9d7d5
3  159  500  48.4660  193  209  204  3  159  500  48.4660  #c1d1cc
4    0  249  43.8037  184  203  203  4    0  249  43.8037  #b8cbcb


`ColumnDataSource` 可以直接从 `DataFrame` 做出来：`circle()` 的 `radius` 连到数据源的 `radius` 列、`fill_color` 连到 `color` 列；为了保持纵横比，把 `match_aspect=True`、`aspect_ratio=1` 设上。

In [7]:
fig = figure(match_aspect=True, aspect_ratio=1, x_range=(0, 500), y_range=(0, 500))
source = ColumnDataSource(data=df2)
fig.circle(
    "x", "y", radius="radius", fill_color="color", source=source, line_color=None
)
show(fig)

### 线段(`Segment`)

用 `Segment` 能画很多颜色与粗细各不相同的线段：`x0`、`x1`、`y0`、`y1`、`line_color`、`line_alpha`、`line_width` 都能连到数据源。下面的例子显示用电子琴录下的 MIDI 数据。

`for_elise.csv` 里是琴键的事件，一行一个事件：`note` 是音、`velocity` 是强弱、`start` 与 `end` 是按下的时刻与放开的时刻。读进 `DataFrame` 之后把音变成文字(76 就是 `"E5"`)；强弱是 0〜127，乘 2 就落在色表 `Viridis256` 的编号范围里。

In [8]:
from bokeh.palettes import Viridis256

notes = "C C# D D# E F F# G G# A A# B".split()
all_notes_name = pd.Series([f"{note}{i-1}" for i in range(0, 15) for note in notes])
df = pd.read_csv("data/for_elise.csv")
df["note_name"] = all_notes_name[df.note].values
df["color"] = [Viridis256[i * 2] for i in df.velocity]
print(df.head(3))

   note  velocity  start    end note_name    color
0    76        65  0.391  0.506        E5  #1F928C
1    75        74  0.495  0.610       D#5  #1FA386
2    76        71  0.581  0.713        E5  #1E9D88


下面是绘制的部分：❶ 给 `figure()` 的 `y_range` 传列表，y 轴的刻度就用那些值。❷ 用 `Segment` 表示音，`x0`、`y0` 是起点、`x1`、`y1` 是终点。❸ 加上鼠标移上去就显示信息的 `HoverTool`：`tooltips` 的每个元素是元组，第 1 个是名字，第 2 个如果是以 `@` 开头的列名，就显示那一列的值。

In [9]:
from bokeh.models import HoverTool

fig = figure(
    y_range=all_notes_name.loc[df.note.min() : df.note.max()], width=800, height=400
)  # ❶
source = ColumnDataSource(data=df)
fig.segment(
    x0="start",
    x1="end",
    y0="note_name",
    y1="note_name",
    color="color",
    source=source,
    line_width=6,
)  # ❷
hover = HoverTool(tooltips=[("note", "@note_name"), ("velocity", "@velocity")])  # ❸
fig.tools.append(hover)
show(fig)

### 多条折线(`MultiLine`)

`Line` 只能画一条折线，要画很多条就在循环里调 `figure.line()`、一条条做数据源，但数量一多就慢，于是有了能一次画完的 `MultiLine`：`xs`、`ys`、`line_color`、`line_width`、`line_alpha` 都能连到数据源，其中 `xs` 与 `ys` 是嵌套的列表，一组就是一条坐标的排列。

下面的程序画一个长方形与一个直角三角形，线宽与颜色由数据源决定。

In [10]:
source1 = ColumnDataSource(
    data=dict(
        xs=[[0, 1, 1, 0, 0], [0.1, 0.9, 0.5, 0.1]],
        ys=[[0, 0, 1, 1, 0], [0.1, 0.1, 0.5, 0.1]],
        color=["red", "green"],
        width=[2, 4],
    )
)
fig1 = figure(frame_width=300, frame_height=300, toolbar_location=None)
fig1.multi_line("xs", "ys", line_color="color", line_width="width", source=source1);

坐标的排列里有 NaN 时，折线会在那里断开，变成几条线：下面的 `source2` 里 `xs`、`ys` 只有一个元素，却分成 2 条；本来只想画 1 条，`color` 与 `width` 的列表也就只有一个元素。

In [11]:
source2 = ColumnDataSource(
    data=dict(
        xs=[[0, 1, 1, 0, 0, np.nan, 0.1, 0.9, 0.5, 0.1]],
        ys=[[0, 0, 1, 1, 0, np.nan, 0.1, 0.1, 0.5, 0.1]],
        color=["red"],
        width=[2],
    )
)
fig2 = figure(frame_width=300, frame_height=300, toolbar_location=None)
fig2.multi_line("xs", "ys", line_color="color", line_width="width", source=source2)
layout = row(fig1, fig2)
show(layout)

这是用 `MultiLine` 画等高线的例子：Bokeh 没有计算等高线的功能，这里用 `contourpy` 来算。

同一个值的等高线有时会分成好几段曲线，因此用 `np.vstack()` 把坐标接起来，在曲线的断口处插进 NaN。数据源里放 3 项内容。

* `"xs"`：等高线的 x 坐标的列表
* `"ys"`：等高线的 y 坐标的列表
* `"levels"`：各条等高线的值

颜色用 `linear_cmap()`，把 `levels` 的值对应到 `viridis` 的颜色。

In [12]:
from bokeh.transform import linear_cmap
from bokeh.palettes import viridis
import contourpy

Y, X = np.mgrid[-6:6:100j, -8:8:150j]
Z = np.sin(X) + np.cos(Y)

contour_gen = contourpy.contour_generator(X, Y, Z, name="serial")

level_values = np.linspace(Z.min(), Z.max(), 10)[1:-1]
levels_lines = contour_gen.multi_lines(level_values)

xs = []
ys = []
for level in levels_lines:
    lines = []
    for line in level:
        lines.append(line)
        lines.append([[np.nan, np.nan]])
    lines = np.vstack(lines)
    xs.append(lines[:, 0])
    ys.append(lines[:, 1])

source = ColumnDataSource(data={"xs": xs, "ys": ys, "levels": level_values})

fig = figure(
    match_aspect=True,
    aspect_ratio=8 / 6,
    x_range=(X.min(), X.max()),
    y_range=(Y.min(), Y.max()),
)
cmap = linear_cmap("levels", viridis(256), Z.min(), Z.max())
lines = fig.multi_line("xs", "ys", line_color=cmap, source=source, line_width=2)
show(fig)

### 多边形(`Patch` 与 `Patches`)

与 `Line` 和 `MultiLine` 的关系一样，`Patch` 画一个填色的多边形、`Patches` 画多个。`Patches` 的属性与 `MultiLine` 相近，多了填色与花纹的 `fill_alpha`、`fill_color`、`hatch_alpha`、`hatch_color` 等。下面的例子用 `Patches` 画出中国各个省的形状。

In [13]:
import json
from bokeh.models import ColumnDataSource, CustomJS, HoverTool, Div
from bokeh.plotting import figure
from bokeh.transform import linear_cmap
from bokeh.palettes import viridis

with open("data/china_simple.json", encoding="utf-8") as f:
    geo = json.load(f)  # ❶

xs = []
ys = []
population = []

for feature in geo["features"]:
    population.append(feature["properties"]["population"])  # ❷
    geometry = feature["geometry"]
    geo_type = geometry["type"]
    coordinates = geometry["coordinates"]

    if geo_type == "Polygon":
        x, y = zip(*coordinates[0])
    elif geo_type == "MultiPolygon":
        xys = []
        for item in coordinates:
            xys.extend(item[0])
            xys.append([np.nan, np.nan])
        x, y = zip(*xys)

    xs.append(x)
    ys.append(y)

fig = figure(match_aspect=True, aspect_scale=1, frame_width=500, frame_height=350)
source = ColumnDataSource(data=dict(xs=xs, ys=ys, population=population))
cmap = linear_cmap(
    "population", viridis(256), low=np.min(population), high=np.max(population)
)
patches = fig.patches("xs", "ys", source=source, fill_color=cmap, line_color="black")
show(fig)

Bokeh 里还有在浏览器里读 GeoJSON 的 `GeoJSONDataSource`：把 GeoJSON 的字符串放进 `geojson`，浏览器的 JavaScript 就把它变成列，其中 `xs` 与 `ys` 是多边形的坐标，各个地点的属性也变成列，例如 `population` 列。

In [14]:
from bokeh.models import GeoJSONDataSource

with open("data/china_simple.json", encoding="utf-8") as f:
    geojson_string = f.read()

source = GeoJSONDataSource(geojson=geojson_string)
fig = figure(match_aspect=True, aspect_scale=1, frame_width=500, frame_height=350)
cmap = linear_cmap("population", viridis(256), low=None, high=None)
patches = fig.patches("xs", "ys", source=source, line_color="black", fill_color=cmap)
show(fig)

`GeoJSONDataSource` 的列是在浏览器里做出来的，从 Python 看不到内容。

另外还有画带洞多边形的 `MultiPolygons`，坐标要用多重嵌套的列表来表示，用法请看 Bokeh 的说明。

### 六角形格子(`HexTile`)

`HexTile` 画蜂窝那样的正六角形格子：`fill_alpha`、`fill_color`、`line_alpha`、`line_color`、`line_width`、`q`、`r`、`scale` 等都能连到数据源，`q`、`r` 是格子的坐标。

六角形的排法有两种：顶点朝上(`pointytop`)与平边朝上(`flattop`)。下面的图里蓝色是 Q 轴、橙色是 R 轴，里面的数字是 Q-R 坐标；注意 R 轴的正方向朝下。

In [15]:
from bokeh.palettes import Category10_10
from bokeh.util.hex import axial_to_cartesian
from bokeh.layouts import row

q, r = [a.ravel() for a in np.mgrid[-2:3, -2:3]]
colors = np.array([Category10_10[7]] * len(q))
colors[r == 0] = Category10_10[0]
colors[q == 0] = Category10_10[1]
colors[(r == 0) & (q == 0)] = Category10_10[2]

figs = []
for orientation in ["pointytop", "flattop"]:
    fig = figure(match_aspect=True, toolbar_location=None, title=orientation)
    fig.grid.visible = False
    fig.hex_tile(
        q,
        r,
        size=1,
        line_color="white",
        fill_color=colors,
        alpha=0.5,
        orientation=orientation,
    )
    x, y = axial_to_cartesian(q, r, 1, orientation)
    fig.text(
        x,
        y,
        text=["(%d, %d)" % (q, r) for (q, r) in zip(q, r)],
        text_baseline="middle",
        text_align="center",
        font_size="10pt",
    )
    figs.append(fig)

layout_hex_tile = row(figs)
show(layout_hex_tile)

为了算格子的坐标，有 `cartesian_to_axial()`、`axial_to_cartesian()`、`hexbin()`：`cartesian_to_axial()` 把直角坐标变成格子的坐标，`hexbin()` 数出点落在哪个六角形里。

下面的例子把图像做成六角形的马赛克：❶ 用 `cartesian_to_axial()` 把图像像素的坐标变成格子的坐标，`size` 是六角形外接圆的半径。❷ 把图像变成 `DataFrame`，`q`、`r` 是格子的坐标，`red`、`green`、`blue` 是颜色的分量。❸ 用 `groupby()` 算出每个六角形的平均颜色。❹ 把平均的颜色变成十六进制的字符串。❺ 用 `hex_tile()` 画时，要传与 `cartesian_to_axial()` 相同的 `size` 与 `orientation`。

In [16]:
from imageio.v3 import imread
from bokeh.util.hex import cartesian_to_axial

img = imread("data/vinci_target.png")
h, w, _ = img.shape

size = 8
Y, X = np.mgrid[:h, :w]
Q, R = cartesian_to_axial(X, Y, size, orientation="pointytop")  # ❶

df_img = pd.DataFrame(
    np.dstack([Q, R, img[::-1]]).reshape(-1, 5),
    columns=["q", "r", "red", "green", "blue"],
)  # ❷
df_img2 = df_img.groupby(["q", "r"]).mean().astype(np.uint32).reset_index()  # ❸

tile_colors = (
    df_img2.eval("red*2**16 + green*2**8 + blue").map("#{:06x}".format).tolist()
)  # ❹

source = ColumnDataSource(
    data=dict(q=df_img2.q.values, r=df_img2.r.values, color=tile_colors)
)
fig = figure(
    match_aspect=True,
    frame_width=w,
    frame_height=h,
    x_range=(0, w),
    y_range=(0, h),
    toolbar_location=None,
)
fig.xaxis.visible = fig.yaxis.visible = False
renderer = fig.hex_tile(
    "q",
    "r",
    fill_color="color",
    line_color=None,
    size=size,
    orientation="pointytop",
    source=source,
)  # ❺
show(fig);

### 文字(`Text`)

`Text` 用来往图里加注释：`angle`、`text`、`text_alpha`、`text_color`、`text_font_size`、`x`、`y`、`x_offset`、`y_offset` 等都能连到数据源，`x`、`y` 是文字的位置，位置与文字的关系由 `text_align` 与 `text_baseline` 决定。下面的图是它们的候选值以及与文字的位置关系。

In [17]:
# %hide
from bokeh.core import enums
from itertools import product

text_align = list(enums.TextAlign)
text_baseline = list(enums.TextBaseline)

fig = figure(
    height=400,
    x_range=["left", "center", "right"],
    y_range=["bottom", "ideographic", "alphabetic", "middle", "hanging", "top"],
    toolbar_location=None,
)
fig.xaxis.axis_label = "text_align"
fig.yaxis.axis_label = "text_baseline"

for align, baseline in product(text_align, text_baseline):
    fig.text(
        [align],
        [baseline],
        ["Py³"],
        font_size="30pt",
        text_align=align,
        text_baseline=baseline,
    )

show(fig)

下面的例子用 `Text` 画词云：`python_words.csv` 是用 `wordcloud` 库做出的数据，统计的是 Python 标准库源码里常见的词。`x`、`y` 是文字的位置，对齐方式是 `'center'` 与 `'bottom'`，`size` 是文字的大小，要像 `'32px'` 这样带单位，`px` 表示像素；`angle` 是文字的朝向，单位是弧度。

|  x |  y | word | size | angle | color |
|---:|---:|:--|:--|---:|:--|
| 239 | 453 | return | 32px | 0 | #84adcd |
| 201 | 293 | header | 11px | 0 | #3975a5 |
| 158.5 | 213 | val | 6px | 0 | #ffe873 |
| 285 | 424.5 | process | 6px | 1.5708 | #3b77a8 |
| 63 | 348 | del | 11px | 0 | #498abd |

文字的大小是像素，数据的坐标也取像素：给 `figure()` 的 `frame_height` 与 `frame_width` 传像素，再把 `x_range` 与 `y_range` 设成同样的大小，1 个单位就是 1 像素。

In [18]:
df = pd.read_csv("data/python_words.csv")
from bokeh.core.property.vectorization import Value

fig = figure(
    frame_height=500,
    frame_width=500,
    x_range=(0, 500),
    y_range=(0, 500),
    toolbar_location=None,
)
fig.grid.visible = False
fig.background_fill_color = "black"
source = ColumnDataSource(data=df)
fig.text(
    "x",
    "y",
    text="word",
    angle="angle",
    text_font_size="size",
    text_color="color",
    text_align="center",
    text_baseline="bottom",
    text_font_style="normal",
    text_font=Value("times"),
    source=source,
)

fig.xaxis.visible = False
fig.yaxis.visible = False
show(fig)

### 图像

Bokeh 里有 3 个用于显示图像的图形：`Image` 与 `ImageRGBA` 能连 `dh`、`dw`、`image`、`x`、`y` 等，`ImageURL` 能连 `angle`、`h`、`url`、`w`、`x`、`y` 等。`x`、`y` 是左下角的坐标，`dh`、`dw` 是数据坐标里的高与宽。

* `Image`：把二维数组用调色板或颜色映射器变成图像，适合显示双变量的函数。
* `ImageRGBA`：把形状 `(高, 宽, 4)` 的数组变成红、绿、蓝、透明度的图像。
* `ImageURL`：用 URL 指定图像。

下面的例子用这 3 个显示同一个二维函数，`Image` 要传 `palette`。

In [19]:
from bokeh.palettes import viridis

Y, X = np.ogrid[-3:3:100j, -3:3:100j]
Z = np.sin(X) + np.sin(Y)

x_min, y_min, x_ptp, y_ptp = X.min(), Y.min(), np.ptp(X), np.ptp(Y)
source1 = ColumnDataSource(
    data=dict(
        image=[Z.astype(np.float32)], x=[x_min], y=[y_min], dw=[x_ptp], dh=[y_ptp]
    )
)
fig1 = figure(
    frame_width=300, match_aspect=True, aspect_ratio=x_ptp / y_ptp, title="Image"
)
fig1.image("image", "x", "y", "dw", "dh", source=source1, palette=viridis(256));

`ImageRGBA` 的图像数据是 32 位整数的二维数组：下面的代码用 `matplotlib.cm` 把归一化后的 `Zn` 变成 RGBA 的数组 `Z_img`，再用 `view()` 变成 `uint32` 的 `Z_img_show`。

In [20]:
from matplotlib import cm

Zn = (Z - Z.min()) / np.ptp(Z)
Z_img = cm.viridis(Zn, bytes=True)
Z_img_show = Z_img.view(np.uint32)[:, :, 0]
source2 = ColumnDataSource(
    data=dict(image=[Z_img_show], x=[x_min], y=[y_min], dw=[x_ptp], dh=[y_ptp])
)
fig2 = figure(
    frame_width=300, match_aspect=True, aspect_ratio=x_ptp / y_ptp, title="ImageRGBA"
)
fig2.image_rgba("image", "x", "y", "dw", "dh", source=source2);

`Image` 与 `ImageRGBA` 会把 Python 的数组原样送到浏览器，数组一大传输量就上去了，因此在 Python 里先压成 JPEG、做成数据 URL，再用 `ImageURL` 显示。本书 `helper.bokeh` 的 `encode_image()` 就做这个转换(用 OpenCV)：图像的原点在左上角，这里先把 `Z_img` 上下翻转再压成 JPEG；`ImageURL` 的 `anchor` 决定图像原点的位置。

In [21]:
from helper.bokeh import encode_image

Z_jpeg = encode_image(Z_img[::-1])
source3 = ColumnDataSource(
    data=dict(image=[Z_jpeg], x=[x_min], y=[y_min], dw=[x_ptp], dh=[y_ptp])
)
fig3 = figure(
    frame_width=300, match_aspect=True, aspect_ratio=x_ptp / y_ptp, title="ImageURL"
)
fig3.image_url("image", "x", "y", "dw", "dh", source=source3, anchor="bottom_left");

In [22]:
from bokeh.layouts import row

layout = row(fig1, fig2, fig3)
show(layout)

比较 3 个图写成 HTML 时的大小：`Image` 是 32 位浮点数、`ImageRGBA` 是 32 位整数，大小因此相近；压成 JPEG 的 `ImageURL` 小得多。

In [23]:
from bokeh import embed

for name, fig in zip(("image", "image_rgba", "image_url"), [fig1, fig2, fig3]):
    size = len(embed.file_html(fig, resources="cdn"))
    print(f"{name:10s} = {size:>6d}")

image      =  50615
image_rgba =  19850
image_url  =   8621


## 小结

* 图形用 `figure` 的方法来做，属性可以连到数据源的列。
* 类型名带 `Spec` 的属性，能接受值、列名或变换器。
* 用 `factor_mark()`、`factor_cmap()`、`linear_cmap()` 把数据变成形状或颜色。
* `Circle` 能用数据的单位决定半径，用 `match_aspect` 与 `radius_dimension` 调整。
* 多条线用 `MultiLine`、多边形用 `Patches`、带洞的用 `MultiPolygons`，断口用 NaN 表示。
* 地图用 `GeoJSONDataSource` 读，列由浏览器来建。
* `HexTile` 画六角形格子、`Text` 加注释，`Image`、`ImageRGBA`、`ImageURL` 显示图像。